# P1 Kaggle H100 IRIS 3B

- Change class: `Capability expansion`
- Mandatory docs consulted: `docs/00,01,02,03,04,05,06,07,08,09,10,13,14,15,16,17,18,19`
- Checkpoint repo: `Danwuoo/IRIS-math`
- Final release repo: set `HF_FINAL_REPO_ID` below before the final export cell

This notebook is organized for repeated 6-hour Kaggle H100 cycles:
1. bootstrap environment and cache roots under `/kaggle/temp`
2. resume from `Danwuoo/IRIS-math` if a prior cycle exists
3. run one governed training cycle
4. optionally rebuild S8 paths, run Phase D/E gates, and append the P1 readiness packet
5. only finalize the HF release after the readiness packet reports the third consecutive PASS


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_REPO_DIR = Path('/kaggle/input/datasets/danielwuu/iris-math/packages/')
KAGGLE_WHEEL_DIR = Path('/kaggle/input/datasets/danielwuu/wheels/wheels/linux_cp312/')
WORK_REPO = Path('/kaggle/working/IRIS-math')
CACHE_ROOT = Path('/kaggle/temp/iris_p1_cache')
RUN_ROOT = Path('/kaggle/temp/iris_p1_run')
TOKENIZER_WORKDIR = Path('/kaggle/working/iris_p1_tokenizer')
SNAPSHOT_ROOT = Path('/kaggle/temp/iris_p1_snapshots')
HF_CHECKPOINT_REPO_ID = 'Danwuoo/IRIS-math'
HF_FINAL_REPO_ID = os.environ.get('HF_FINAL_REPO_ID', '')
HF_SECRET_LABEL = os.environ.get('HF_SECRET_LABEL', 'HF_TOKEN')
RUN_ID = os.environ.get('IRIS_RUN_ID', 'p1-iris3b-kaggle')
MAX_CYCLE_MINUTES = int(os.environ.get('MAX_CYCLE_MINUTES', '350'))
DATASET_CACHE_LIMIT_GIB = int(os.environ.get('DATASET_CACHE_LIMIT_GIB', '50'))
STREAMING_MODE = os.environ.get('IRIS_STREAMING_MODE', 'auto')
P1_MANIFEST_PATH = os.environ.get('P1_MANIFEST_PATH', '').strip()
BASELINE_ID = os.environ.get('P1_BASELINE_ID', 'p1-readiness-fixed-baseline')
TOLERANCE_PROFILE_ID = os.environ.get('P1_TOLERANCE_PROFILE_ID', 'tp_p1_bootstrap')
READINESS_BASELINE_MODE = os.environ.get('READINESS_BASELINE_MODE', 'auto')

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret(HF_SECRET_LABEL)
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise RuntimeError(
        'HF token not found. Put it in Kaggle Notebook Secrets as HF_TOKEN, '
        'or set environment variable HF_SECRET_LABEL to your custom secret name.'
    )

for path in (CACHE_ROOT, RUN_ROOT, TOKENIZER_WORKDIR, SNAPSHOT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = str(CACHE_ROOT / 'hf')
os.environ['HF_HUB_CACHE'] = str(CACHE_ROOT / 'hf' / 'hub')
os.environ['HF_DATASETS_CACHE'] = str(CACHE_ROOT / 'hf' / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_ROOT / 'hf' / 'transformers')

if not WORK_REPO.exists():
    if not KAGGLE_REPO_DIR.exists():
        raise FileNotFoundError(f'Missing repo input: {KAGGLE_REPO_DIR}')
    shutil.copytree(KAGGLE_REPO_DIR, WORK_REPO)

DEFAULT_P1_MANIFEST_PATH = WORK_REPO / 'src' / 'iris' / 'train' / 'data' / 'profiles' / 'p1_streaming_manifest_v1.json'
if not P1_MANIFEST_PATH:
    P1_MANIFEST_PATH = str(DEFAULT_P1_MANIFEST_PATH)

os.chdir(WORK_REPO)
print(json.dumps({
    'WORK_REPO': str(WORK_REPO),
    'RUN_ROOT': str(RUN_ROOT),
    'CACHE_ROOT': str(CACHE_ROOT),
    'HF_CHECKPOINT_REPO_ID': HF_CHECKPOINT_REPO_ID,
    'HF_FINAL_REPO_ID': HF_FINAL_REPO_ID,
    'HF_SECRET_LABEL': HF_SECRET_LABEL,
    'STREAMING_MODE': STREAMING_MODE,
    'P1_MANIFEST_PATH': P1_MANIFEST_PATH,
    'DATASET_CACHE_LIMIT_GIB': DATASET_CACHE_LIMIT_GIB,
    'MAX_CYCLE_MINUTES': MAX_CYCLE_MINUTES,
}, indent=2))


In [ ]:
def run_cmd(args: list[str], *, cwd: Path | None = None, env: dict | None = None, check: bool = True):
    cmd = [str(part) for part in args]
    print('$', ' '.join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd or WORK_REPO),
        env=env or os.environ.copy(),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed: {proc.returncode}')
    return proc


if not KAGGLE_WHEEL_DIR.exists():
    raise FileNotFoundError(f'Missing wheel input: {KAGGLE_WHEEL_DIR}')
JAX_REQUIREMENTS = WORK_REPO / 'wheels' / 'requirements_extra_jax_cuda12.txt'
if not JAX_REQUIREMENTS.exists():
    raise FileNotFoundError(f'Missing JAX requirement manifest: {JAX_REQUIREMENTS}')

run_cmd([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(KAGGLE_WHEEL_DIR), '--upgrade', 'pip'])
run_cmd([
    sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(KAGGLE_WHEEL_DIR),
    '-r', str(JAX_REQUIREMENTS),
    'datasets', 'transformers', 'sentencepiece', 'huggingface-hub'
])


In [ ]:
cycle_args = [
    sys.executable,
    'scripts/train_p1_3b.py',
    'cycle',
    '--output-dir', str(RUN_ROOT),
    '--run-id', RUN_ID,
    '--hf-checkpoint-repo-id', HF_CHECKPOINT_REPO_ID,
    '--hf-token', HF_TOKEN,
    '--streaming-mode', STREAMING_MODE,
    '--snapshot-root', str(SNAPSHOT_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--dataset-cache-limit-gib', str(DATASET_CACHE_LIMIT_GIB),
    '--tokenizer-workdir', str(TOKENIZER_WORKDIR),
    '--max-cycle-minutes', str(MAX_CYCLE_MINUTES),
    '--baseline-id', BASELINE_ID,
    '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
]
if P1_MANIFEST_PATH:
    cycle_args.extend(['--manifest-path', P1_MANIFEST_PATH])
cycle_proc = run_cmd(cycle_args)
cycle_summary = json.loads([line for line in cycle_proc.stdout.splitlines() if line.strip()][-1])
print(json.dumps(cycle_summary, indent=2))


In [ ]:
S8_ROOT = RUN_ROOT / 's8_paths'
S8_PATHS = {
    'uninterrupted': S8_ROOT / 'uninterrupted',
    'execute_crash': S8_ROOT / 'execute_crash',
    'pre_commit_crash': S8_ROOT / 'pre_commit_crash',
    'post_commit_crash': S8_ROOT / 'post_commit_crash',
}

def with_manifest_path(args: list[str]) -> list[str]:
    if P1_MANIFEST_PATH:
        return [*args, '--manifest-path', P1_MANIFEST_PATH]
    return args

def rebuild_s8_paths(max_cycle_minutes: int = 10) -> dict[str, str]:
    for path in S8_PATHS.values():
        shutil.rmtree(path, ignore_errors=True)
    pinned_runtime_lock = RUN_ROOT / 'runtime_lock_manifest.json'

    run_cmd(with_manifest_path([
        sys.executable, 'scripts/train_p1_3b.py', 'cycle',
        '--output-dir', str(S8_PATHS['uninterrupted']),
        '--run-id', f'{RUN_ID}-s8-uninterrupted',
        '--hf-checkpoint-repo-id', HF_CHECKPOINT_REPO_ID,
        '--hf-token', HF_TOKEN,
        '--streaming-mode', STREAMING_MODE,
        '--snapshot-root', str(SNAPSHOT_ROOT),
        '--cache-root', str(CACHE_ROOT),
        '--dataset-cache-limit-gib', str(DATASET_CACHE_LIMIT_GIB),
        '--tokenizer-workdir', str(TOKENIZER_WORKDIR),
        '--runtime-lock-manifest', str(pinned_runtime_lock),
        '--max-cycle-minutes', str(max_cycle_minutes),
        '--max-segments', '1',
        '--no-download-latest',
        '--no-sync-checkpoint',
    ]))

    for crash_point, resume_path_id in [('execute', 'execute_crash'), ('pre_commit', 'pre_commit_crash'), ('post_commit', 'post_commit_crash')]:
        crash_dir = S8_PATHS[resume_path_id]
        run_cmd(with_manifest_path([
            sys.executable, 'scripts/train_p1_3b.py', 'cycle',
            '--output-dir', str(crash_dir),
            '--run-id', f'{RUN_ID}-{resume_path_id}',
            '--hf-checkpoint-repo-id', HF_CHECKPOINT_REPO_ID,
            '--hf-token', HF_TOKEN,
            '--streaming-mode', STREAMING_MODE,
            '--snapshot-root', str(SNAPSHOT_ROOT),
            '--cache-root', str(CACHE_ROOT),
            '--dataset-cache-limit-gib', str(DATASET_CACHE_LIMIT_GIB),
            '--tokenizer-workdir', str(TOKENIZER_WORKDIR),
            '--runtime-lock-manifest', str(pinned_runtime_lock),
            '--max-cycle-minutes', str(max_cycle_minutes),
            '--max-segments', '1',
            '--no-download-latest',
            '--no-sync-checkpoint',
            '--crash-point', crash_point,
            '--crash-segment', '0',
        ]), check=False)
        run_cmd(with_manifest_path([
            sys.executable, 'scripts/train_p1_3b.py', 'cycle',
            '--output-dir', str(crash_dir),
            '--run-id', f'{RUN_ID}-{resume_path_id}',
            '--hf-checkpoint-repo-id', HF_CHECKPOINT_REPO_ID,
            '--hf-token', HF_TOKEN,
            '--streaming-mode', STREAMING_MODE,
            '--snapshot-root', str(SNAPSHOT_ROOT),
            '--cache-root', str(CACHE_ROOT),
            '--dataset-cache-limit-gib', str(DATASET_CACHE_LIMIT_GIB),
            '--tokenizer-workdir', str(TOKENIZER_WORKDIR),
            '--runtime-lock-manifest', str(pinned_runtime_lock),
            '--max-cycle-minutes', str(max_cycle_minutes),
            '--max-segments', '1',
            '--no-download-latest',
            '--no-sync-checkpoint',
            '--resume-path-id', resume_path_id,
        ]))
    return {key: str(value) for key, value in S8_PATHS.items()}

print('Call rebuild_s8_paths() before Phase D/E if you want governed S8 path artifacts instead of reusing RUN_ROOT.')


In [ ]:
PHASE_D_DIR = RUN_ROOT / 'phase_d'
PHASE_E_DIR = RUN_ROOT / 'phase_e'
PHASE_D_BASELINE_DIR = RUN_ROOT / 'phase_d_baseline'
PHASE_E_BASELINE_DIR = RUN_ROOT / 'phase_e_baseline'
READINESS_DIR = RUN_ROOT / 'readiness'
READINESS_DIR.mkdir(parents=True, exist_ok=True)
READINESS_BASELINE_PACKET = READINESS_DIR / 'p1_baseline_snapshot.json'
READINESS_HISTORY_PATH = READINESS_DIR / 'p1_readiness_history.jsonl'

effective_s8_paths = {key: value for key, value in S8_PATHS.items() if value.exists()}
if not effective_s8_paths:
    print('No governed S8 paths found; Phase D/E will reuse RUN_ROOT for all four path arguments.')
    effective_s8_paths = {key: RUN_ROOT for key in S8_PATHS}

phase_d_common = [
    sys.executable, 'scripts/phase_d_gate.py',
    '--phase', 'D',
    '--model-run-dir', str(RUN_ROOT),
    '--baseline-report-dir', str(PHASE_D_BASELINE_DIR),
    '--output-dir', str(PHASE_D_DIR),
    '--baseline-id', BASELINE_ID,
    '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
    '--device', 'cpu',
    '--h100-uninterrupted', str(effective_s8_paths['uninterrupted']),
    '--h100-execute', str(effective_s8_paths['execute_crash']),
    '--h100-pre-commit', str(effective_s8_paths['pre_commit_crash']),
    '--h100-post-commit', str(effective_s8_paths['post_commit_crash']),
]
phase_e_common = [
    sys.executable, 'scripts/phase_e_gate.py',
    '--phase', 'E',
    '--model-run-dir', str(RUN_ROOT),
    '--baseline-report-dir', str(PHASE_E_BASELINE_DIR),
    '--output-dir', str(PHASE_E_DIR),
    '--baseline-id', BASELINE_ID,
    '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
    '--device', 'cpu',
    '--h100-uninterrupted', str(effective_s8_paths['uninterrupted']),
    '--h100-execute', str(effective_s8_paths['execute_crash']),
    '--h100-pre-commit', str(effective_s8_paths['pre_commit_crash']),
    '--h100-post-commit', str(effective_s8_paths['post_commit_crash']),
]

freeze_baseline = READINESS_BASELINE_MODE == 'freeze' or not READINESS_BASELINE_PACKET.exists()
if freeze_baseline:
    run_cmd(phase_d_common + ['--freeze-baseline'])
    run_cmd(phase_e_common + ['--freeze-baseline'])
    run_cmd([
        sys.executable, 'scripts/p1_readiness_packet.py',
        '--phase-d-dir', str(PHASE_D_DIR),
        '--phase-e-dir', str(PHASE_E_DIR),
        '--model-run-dir', str(RUN_ROOT),
        '--output-dir', str(READINESS_DIR),
        '--baseline-id', BASELINE_ID,
        '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
        '--freeze-baseline',
    ])
else:
    run_cmd(phase_d_common)
    run_cmd(phase_e_common)
    run_cmd([
        sys.executable, 'scripts/p1_readiness_packet.py',
        '--phase-d-dir', str(PHASE_D_DIR),
        '--phase-e-dir', str(PHASE_E_DIR),
        '--model-run-dir', str(RUN_ROOT),
        '--output-dir', str(READINESS_DIR),
        '--baseline-packet', str(READINESS_BASELINE_PACKET),
        '--history-path', str(READINESS_HISTORY_PATH),
        '--baseline-id', BASELINE_ID,
        '--tolerance-profile-id', TOLERANCE_PROFILE_ID,
        '--no-hard-fail',
    ])

packet_path = READINESS_DIR / 'p1_readiness_packet.json'
if packet_path.exists():
    packet = json.loads(packet_path.read_text(encoding='utf-8-sig'))
    print(json.dumps({
        'run_gate_status': packet.get('run_gate_status'),
        'promotion_status': packet.get('promotion_status'),
        'consecutive_gate_passed_runs': packet.get('consecutive_gate_passed_runs'),
    }, indent=2))


In [ ]:
if not HF_FINAL_REPO_ID:
    raise RuntimeError('Set HF_FINAL_REPO_ID before running final export.')

finalize_proc = run_cmd([
    sys.executable,
    'scripts/train_p1_3b.py',
    'finalize',
    '--run-dir', str(RUN_ROOT),
    '--release-dir', str(RUN_ROOT / 'final_release'),
    '--tokenizer-root', str(RUN_ROOT / 'tokenizer' / 'iris_p1_tokenizer'),
    '--streaming-manifest-path', str(RUN_ROOT / 'data' / 'p1_streaming_manifest_committed.json'),
    '--readiness-packet-path', str(READINESS_DIR / 'p1_readiness_packet.json'),
    '--readiness-history-path', str(READINESS_DIR / 'p1_readiness_history.jsonl'),
    '--hf-final-repo-id', HF_FINAL_REPO_ID,
    '--hf-token', HF_TOKEN,
])
finalize_summary = json.loads([line for line in finalize_proc.stdout.splitlines() if line.strip()][-1])
print(json.dumps(finalize_summary, indent=2))
